<a href="https://colab.research.google.com/github/Abooduoto/Agent-7/blob/main/agent_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 ورشة عمل: الوكيل السابع (Agent 7 Workshop)
## From Zero to a Full 7-Agent Workflow using LangGraph

مرحباً بكم في ورشة "الوكيل 7". اليوم سنبني فريقاً متكاملاً (Squad) مكوناً من 7 وكلاء متخصصين، يعملون بتناغم تام لخدمة هدف واحد: **تأمين الوظيفة المثالية لك**.

### لماذا 7 وكلاء؟
في العمل الحقيقي، لا يقوم شخص واحد بكل شيء. نحتاج إلى التخصص:
1. **🕵️‍♂️ الباحث (Researcher):** يمسح السوق ويجد الفرص.
2. **🧠 الاستراتيجي (Strategist):** العقل المدبر الذي يختار ويحلل.
3. **📝 كاتب السيرة (Resume Writer):** يصيغ سيرتك لتلائم الوظيفة.
4. **💌 كاتب الرسائل (Review Writer):** يكتب رسالة تغطية جذابة.
5. **💼 مهندس المعرض (Portfolio Builder):** يقترح مشاريع لإثبات مهاراتك.
6. **📊 موظف العمليات (OPS):** يسجل كل شيء في قاعدة بيانات (CSV).
7. **✅ المدقق (QA):** يراجع الجودة النهائية.

سنبني هذا النظام باستخدام **LangGraph** للتحكم الكامل في تدفق العمليات.

## 🛠️ الدليل العملي للمشاركين (Participants' Practical Guide)
قبل أن نغوص في الكود، هذه خطتك لتجهيز بيئة العمل (Environment Setup) بشكل احترافي.

### 1️⃣ الأدوات والحسابات (Accounts & Tools)
لبناء وكيل ذكي يرى العالم ويفكر، سنحتاج أداتين رئيسيتين:
1.  **المحرك البحثي (The Eyes):** خدمة `Tavily` للبحث المتقدم.
2.  **العقل المدبر (The Brain):** نموذج `Llama 3` عبر `Ollama`.

#### **أولاً: مفتاح Tavily (مجاني)**
هذا المفتاح هو "تصريح العبور" لوكيلك ليدخل الإنترنت.
1.  اذهب إلى [tavily.com](https://app.tavily.com/sign-in).
2.  سجل باستخدام Google أو GitHub.
3.  في الصفحة الرئيسية، ستجد مفتاحك يبدأ بـ `tvly-`. **انسخه!**

#### **ثانياً: حماية المفتاح في Colab (Security Best Practice)**
لا تضع المفتاح أبداً داخل الكود مباشرة (Hardcoding). سنستخدم "خزنة الأسرار":
1.  انظر في يسار الشاشة في Colab، ستجد أيقونة "مفتاح" 🔑 (Secrets).
2.  اضغط عليها ثم اختر **Add new secret**.
3.  **الاسم (Name):** اكتب `TAVILY_API_KEY` (بالحروف الكبيرة).
4.  **القيمة (Value):** ألصق المفتاح الذي نسخته.
5.  ⚠️ **مهم جداً:** فعّل الزر (Toggle) بجانب المفتاح ليصبح أزرق، وتظهر لك رسالة "Notebook access granted".

---

### 2️⃣ المكتبات (Libraries)
سنستخدم أحدث ما توصلت إليه تكنولوجيا الوكلاء:
*   `langgraph`: لبناء هيكل الوكيل (الهيكل العظمي).
*   `langchain` & `langchain_ollama`: لربط الكود بنموذج الذكاء الاصطناعي.
*   `tavily-python`: أداة البحث.

In [ ]:
# تثبيت المكتبات
!pip install -q langgraph langchain langchain_ollama tavily-python

In [ ]:
# تشغيل Ollama (العقل المدبر)
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

process = subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

print("⏳ Downloading Llama 3.1...")
!ollama pull llama3.1
print("✅ Ready! All 7 Agents are waking up...")

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
⏳ Downloading Llama 3.1...

✅ Ready! All 7 Agents are waking up...


In [ ]:
import os
from google.colab import userdata

try:
    os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
    print("✅ Keys Loaded.")
except:
    print("⚠️ Please set TAVILY_API_KEY in Colab Secrets.")

✅ Keys Loaded.


## 1️⃣ الذاكرة المشتركة (The State)
هذا هو "الملف" الذي سيتناقله الوكلاء السبعة.

In [ ]:
from typing import TypedDict, List, Dict

class Agent7State(TypedDict):
    # المدخلات
    job_query: str
    country: str
    candidate_profile: str

    # مخرجات الوكلاء
    job_listings: List[Dict]        # Agent 1 Output
    selected_job: Dict              # Agent 2 Output
    gap_analysis: str               # Agent 2 Output
    resume_md: str                  # Agent 3 Output
    cover_letter_md: str            # Agent 4 Output
    portfolio_md: str               # Agent 5 Output
    ops_status: str                 # Agent 6 Output
    final_report: str               # Agent 7 Output

## 2️⃣ تعريف الوكلاء (Defining the 7 Agents)
سنعرف دالة (Node) لكل وكيل.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from tavily import TavilyClient
import json
import pandas as pd

llm = ChatOllama(model="llama3.1", temperature=0.5)

# --- Agent 1: Researcher ---
def researcher_node(state: Agent7State):
    print("🕵️‍♂️ Agent 1 (Researcher): Scouring the web...")
    tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    query = f"Active {state['job_query']} jobs in {state['country']}"
    res = tavily.search(query=query, max_results=6, search_depth="advanced")
    return {"job_listings": res.get('results', [])}

# --- Agent 2: Strategist ---
def strategist_node(state: Agent7State):
    print("🧠 Agent 2 (Strategist): Analyzing top matches...")
    prompt = ChatPromptTemplate.from_template("""
    Act as a Senior Career Strategist.
    Candidate: {profile}
    Jobs: {jobs}

    Select the single best job fit. Perform a gap analysis.
    Return ONLY JSON: {{'title': '...', 'company': '...', 'url': '...', 'reason': '...', 'missing_skills': '...'}}
    """)
    chain = prompt | llm
    try:
        # Tip: We treat the output as a string and parse it manually to handle potential LLM quirks
        res = chain.invoke({"profile": state['candidate_profile'], "jobs": str(state['job_listings'])})
        # Cleaning JSON markdown if present
        clean_json = res.content.replace('```json', '').replace('```', '').strip()
        selection = json.loads(clean_json)
        return {"selected_job": selection, "gap_analysis": selection.get('missing_skills', 'None')}
    except:
        # If JSON fails, fallback (in production, we'd have retries)
        print("⚠️ Error parsing strategy, picking first job as fallback.")
        return {"selected_job": state['job_listings'][0], "gap_analysis": "Manual Review Needed"}

# --- Agent 3: Resume Writer ---
def resume_node(state: Agent7State):
    print("📝 Agent 3 (Resume Writer): Tailoring the CV...")
    prompt = ChatPromptTemplate.from_template("""
    Write a specific Markdown Resume for:
    Candidate: {profile}
    Job: {job}
    Highlight skills relevant to this specific job description.
    """)
    res = (prompt | llm).invoke({"profile": state['candidate_profile'], "job": str(state['selected_job'])})
    with open("Resume.md", "w") as f: f.write(res.content)
    return {"resume_md": res.content}

# --- Agent 4: Cover Letter Writer ---
def cover_letter_node(state: Agent7State):
    print("💌 Agent 4 (CL Writer): Drafting the pitch...")
    prompt = ChatPromptTemplate.from_template("""
    Write a compelling Cover Letter for {company}.
    Role: {role}
    Reason they should hire the candidate based on: {reason}
    """)
    job = state['selected_job']
    res = (prompt | llm).invoke({"company": job.get('company'), "role": job.get('title'), "reason": job.get('reason', 'Great fit')})
    with open("CoverLetter.md", "w") as f: f.write(res.content)
    return {"cover_letter_md": res.content}

# --- Agent 5: Portfolio Builder ---
def portfolio_node(state: Agent7State):
    print("💼 Agent 5 (Portfolio): Designing project ideas...")
    prompt = ChatPromptTemplate.from_template("""
    Based on the missing skills: {gaps}
    Suggest 1 solid GitHub project the candidate should build to impress {company}.
    Write a mini-README for it.
    """)
    res = (prompt | llm).invoke({"gaps": state['gap_analysis'], "company": state['selected_job'].get('company')})
    with open("Portfolio_Advice.md", "w") as f: f.write(res.content)
    return {"portfolio_md": res.content}

# --- Agent 6: OPS (Tracker) ---
def ops_node(state: Agent7State):
    print("📊 Agent 6 (OPS): Logging application...")
    # Append to CSV
    job = state['selected_job']
    data = {
        "Date": [pd.Timestamp.now()],
        "Company": [job.get('company', 'Unknown')],
        "Title": [job.get('title', 'Unknown')],
        "URL": [job.get('url', 'Unknown')],
        "Status": ["Ready to Apply"]
    }
    df = pd.DataFrame(data)
    file_name = "applications.csv"
    # Check if exists to append or create new
    if os.path.exists(file_name):
        df.to_csv(file_name, mode='a', header=False, index=False)
    else:
        df.to_csv(file_name, index=False)
    return {"ops_status": "Logged successfully"}

# --- Agent 7: QA Reviewer ---
def qa_node(state: Agent7State):
    print("✅ Agent 7 (QA): Final quality check...")
    report = f"""
    # 🕵️‍♂️ Agent 7 Final Report
    - **Job Selected:** {state['selected_job'].get('title')} at {state['selected_job'].get('company')}
    - **Resume:** Generated ({len(state['resume_md'])} chars)
    - **Cover Letter:** Generated ({len(state['cover_letter_md'])} chars)
    - **Portfolio Idea:** Provided
    - **Tracking:** Logged to applications.csv

    **Next Step:** Review the 'Resume.md' and send your application!
    """
    return {"final_report": report}

## 3️⃣ بناء الجراف (Connecting the 7 Nodes)
سنربط الوكلاء بالتسلسل: 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7

In [ ]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(Agent7State)

# 1. Add Nodes
workflow.add_node("researcher", researcher_node)
workflow.add_node("strategist", strategist_node)
workflow.add_node("resume_writer", resume_node)
workflow.add_node("cl_writer", cover_letter_node)
workflow.add_node("portfolio", portfolio_node)
workflow.add_node("ops", ops_node)
workflow.add_node("qa", qa_node)

# 2. Add Edges (Sequential Flow)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "strategist")
workflow.add_edge("strategist", "resume_writer")
workflow.add_edge("resume_writer", "cl_writer")
workflow.add_edge("cl_writer", "portfolio")
workflow.add_edge("portfolio", "ops")
workflow.add_edge("ops", "qa")
workflow.add_edge("qa", END)

app = workflow.compile()

## 4️⃣ الانطلاق! (Launch Agent 7) 🚀
أدخل بياناتك ودع الفريق يعمل من أجلك.

In [ ]:
inputs = {
    "job_query": "Senior Data Scientist",
    "country": "Saudi Arabia",
    "candidate_profile": """
    Name: Omar Ali
    Experience: 3 Years in Data Analysis.
    Skills: Python, SQL, Tableau.
    Goal: Transitioning to AI/ML roles.
    """
}

print("🟢 Starting Agent 7 Workflow...")
result = app.invoke(inputs)

from IPython.display import Markdown
Markdown(result['final_report'])

🟢 Starting Agent 7 Workflow...
🕵️‍♂️ Agent 1 (Researcher): Scouring the web...
🧠 Agent 2 (Strategist): Analyzing top matches...
